# M7 SVM — 실습 (W11)

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. 1차원 세트피스로 **마진 최대화·서포트 벡터의 지배력**(8→20 불변, 6→5 이동)을 손계산·sklearn으로 완주한다 ⭐
2. blobs로 마진·SV를 시각화하고 **C의 효과**(SV 3 → 7)를 본다
3. **x² 접어 올리기**(0.571 → 1.0)로 커널의 정신을 재현하고, moons에서 rbf(0.956)·**gamma 극단**(train 1.0/test 0.5)을 확인한다 ⭐

**7단계 멘탈모델 초점:** 모델 — 마진 최대의 경계

## Part A. 세트피스 — 세상에서 가장 작은 SVM ⭐
1차원 점 4개: A = {1, 2} / B = {6, 8}. 후보 경계 t=3/4/5의 마진을 먼저 종이에서 계산해 보고(min(왼쪽 최근접, 오른쪽 최근접)), 코드로 검산하세요.

In [ ]:
import numpy as np                                     # 수치 계산

for t in (3, 4, 5):                                    # 후보 대결
    margin = min(t - 2, ___ - t)                       # ✍️ 빈칸: B쪽 최전선(서포트 벡터가 될 그 수)까지의 거리
    print(f't={t}: 마진 = {margin}')                    # 1 / 2(최대!) / 1
print('최적 경계 =', (2 + 6) / 2)                       # 4.0 — 정확히 가운데

from sklearn.svm import SVC                            # sklearn 검산
X1 = np.array([[1.0], [2.0], [6.0], [8.0]])            # 점 4개(1차원)
y1 = np.array([0, 0, 1, 1])                            # A=0, B=1
m = SVC(kernel='linear', C=1000).fit(X1, ___)          # ✍️ 빈칸: 정답 라벨
print('sklearn 경계:', round(-m.intercept_[0] / m.coef_[0][0], 4))  # 4.0 — 일치?
print('서포트 벡터:', m.support_vectors_.ravel().tolist())          # [2, 6]

X_far = np.array([[1.0], [2.0], [6.0], [20.0]])        # 지배력 실험 1: 8 → 20
m_far = SVC(kernel='linear', C=1000).fit(X_far, y1)
print('8→20 이동 후 경계:', round(-m_far.intercept_[0] / m_far.coef_[0][0], 4))  # 4.0 그대로!
X_mv = np.array([[1.0], [2.0], [5.0], [8.0]])          # 지배력 실험 2: 6 → 5
m_mv = SVC(kernel='linear', C=1000).fit(X_mv, y1)
print('6→5 이동 후 경계:', round(-m_mv.intercept_[0] / m_mv.coef_[0][0], 4))     # 3.5로 이동!

> **검산 포인트:** 마진 1/**2**/1 → 최적 t=4=(2+6)/2, SV=[2, 6] — sklearn 일치. **지배력:** 비SV(8)는 20으로 옮겨도 경계 **4.0 그대로**(투명인간), SV(6)를 5로 옮기면 **3.5로 이동**. 로지스틱은 모든 점을, SVM은 **최전선 몇 점만** 본다 — 여백(마진)도 성능(M2a의 기하학).

## Part B. 마진·서포트 벡터 시각화 + C의 효과
2차원 blobs에서 경계(실선)·마진(점선)·SV(테두리)를 그리고, C를 낮추면 SV가 어떻게 변하는지 봅니다.

In [ ]:
import matplotlib.pyplot as plt                        # 그래프
from sklearn.datasets import make_blobs                # 인공 데이터

Xb, yb = make_blobs(n_samples=60, centers=2,           # 두 덩어리 60점
                    random_state=6, cluster_std=1.1)
clf = SVC(kernel='linear', C=1000).fit(Xb, yb)         # 거의 하드 마진

plt.scatter(Xb[:, 0], Xb[:, 1], c=yb, cmap='coolwarm', s=30)
ax = plt.gca()
xx = np.linspace(*ax.get_xlim(), 200)
yy = np.linspace(*ax.get_ylim(), 200)
XX, YY = np.meshgrid(xx, yy)
Z = clf.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)  # 경계까지의 부호 거리
ax.contour(XX, YY, Z, levels=[-1, 0, 1], linestyles=['--', '-', '--'], colors='k')  # 마진·경계
sv = clf.___                                           # ✍️ 빈칸: 서포트 벡터 좌표 속성
ax.scatter(sv[:, 0], sv[:, 1], s=140, facecolors='none', edgecolors='k', linewidths=1.5)
plt.title('SVM margin & support vectors')              # 제목(영어)
plt.show()
print('서포트 벡터 개수(C=1000):', len(sv))             # 3개 — 이 셋이 경계를 결정

for C in (1000, ___):                                  # ✍️ 빈칸: 너그러운(작은) C — 본문의 그 수
    mc = SVC(kernel='linear', C=C).fit(Xb, yb)         # 소프트 마진 비교
    print(f'C={C}: SV {len(mc.support_vectors_)}개')    # 3개 → 7개

> **관찰:** SV **3개**만 테두리 — 나머지 57점은 경계에 무관. **C를 0.1로** 낮추면(너그러운 소프트 마진) SV가 **7개**로 — 더 많은 점이 마진에 관여하며 이상치 한 점에 덜 휘둘림. C는 침범 벌점의 세기 — 교차검증으로(M2b).

## Part C. 접어 올리기 — x² 하나로 0.571 → 1.0 ⭐
A = {−3, −2, 2, 3}, B = {−1, 0, 1} — B가 A 사이에 끼어 직선(문턱 하나)로는 불가. **x²라는 새 축**을 추가하면?

In [ ]:
x_all = np.array([-3.0, -2.0, 2.0, 3.0, -1.0, 0.0, 1.0])  # 7점(1차원)
y_all = np.array([0, 0, 0, 0, 1, 1, 1])                # A=0(바깥), B=1(가운데)

lin1d = SVC(kernel='linear', C=1000).fit(x_all.reshape(-1, 1), y_all)  # 1차원 직선 시도
print('1차원 직선 train 정확도:', round(lin1d.score(x_all.reshape(-1, 1), y_all), 3))  # 0.571 — 불가!

X_fold = np.column_stack([x_all, x_all ** ___])        # ✍️ 빈칸: 접어 올릴 새 축 — 몇 제곱?
lin2d = SVC(kernel='linear', C=1000).fit(X_fold, y_all)  # 2차원에서 직선(=수평선)
print('x² 추가 후 train 정확도:', round(lin2d.score(X_fold, y_all), 3))  # 1.0 — 완벽 분리!

rbf1d = SVC(kernel='rbf', C=1000).fit(x_all.reshape(-1, 1), y_all)  # RBF는 접기 내장
print('RBF(1차원 그대로):', round(rbf1d.score(x_all.reshape(-1, 1), y_all), 3))  # 1.0

fig, axes = plt.subplots(1, 2, figsize=(11, 4))        # 접기 전/후
axes[0].scatter(x_all, np.zeros_like(x_all), c=y_all, cmap='coolwarm', s=60)
axes[0].set_title('1D: no single threshold works')
axes[0].set_yticks([])
axes[1].scatter(x_all, x_all ** 2, c=y_all, cmap='coolwarm', s=60)
axes[1].axhline(2.5, color='k', linestyle='--')        # 분리 수평선
axes[1].set_title('add x^2: one horizontal line separates')
axes[1].set_xlabel('x'); axes[1].set_ylabel('x^2')     # 축(영어)
plt.tight_layout(); plt.show()

> **관찰:** 1차원 직선 **0.571**(7점 중 4점이 한계) → **x² 추가로 1.0**(A는 위, B는 아래 — 수평선 하나로). **커널 트릭 = 이 "접어 올려 재기"의 계산 지름길**, RBF는 접기 내장(1차원 그대로 1.0). **2학기 예고:** "공간을 바꿔 분리"의 정신 = 2학기 D1의 은닉층(XOR — W14의 M9가 다리).

## Part D. 초승달 — linear vs rbf
직선으로 안 되는 실전형 데이터에서 두 커널을 대결시킵니다.

In [ ]:
from sklearn.datasets import make_moons                # 초승달 데이터
from sklearn.model_selection import train_test_split   # 분할
from sklearn.preprocessing import StandardScaler       # 표준화(SVM은 필수!)
from sklearn.pipeline import make_pipeline             # 배관
from matplotlib.colors import ListedColormap

Xm, ym = make_moons(n_samples=300, noise=0.25, random_state=0)
Xmtr, Xmte, ymtr, ymte = train_test_split(Xm, ym, test_size=0.3, random_state=0)

lin = make_pipeline(StandardScaler(), SVC(kernel='linear')).fit(Xmtr, ymtr)  # 직선
rbf = make_pipeline(StandardScaler(), SVC(kernel=___)).fit(Xmtr, ymtr)       # ✍️ 빈칸: 접기 내장 커널
print('linear:', round(lin.score(Xmte, ymte), 3))       # 0.856 — 꼬리를 자름
print('rbf   :', round(rbf.score(Xmte, ymte), 3))       # 0.956 — 달을 따라 감김

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, mdl, name in zip(axes, [lin, rbf], ['linear', 'rbf']):
    xx, yy = np.meshgrid(np.linspace(Xm[:, 0].min() - 0.5, Xm[:, 0].max() + 0.5, 300),
                         np.linspace(Xm[:, 1].min() - 0.5, Xm[:, 1].max() + 0.5, 300))
    Z = mdl.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#fca5a5', '#93c5fd']))
    ax.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap=ListedColormap(['#dc2626', '#2563eb']),
               edgecolor='k', s=15)
    ax.set_title(f'kernel = {name}')                    # 제목(영어)
plt.tight_layout(); plt.show()

> **관찰:** linear **0.856**(직선이 달의 꼬리를 자름) vs rbf **0.956**(곡선이 달을 따라 감김). 참고: 유방암(30특징·표준화)은 선형만으로 0.982 — 고차원에선 직선으로 가를 여지가 커서 선형이 강할 때가 많음.

## Part E. gamma — 과소 → 적합 → 극단 과적합
gamma(한 점의 영향 반경 — 클수록 좁고 예민)를 키우며 train/test를 봅니다. 마지막 값에서 무슨 일이?

In [ ]:
for gamma in (0.1, 1, 100, ___):                       # ✍️ 빈칸: 극단 과적합을 볼 큰 gamma(본문의 그 수)
    mg = make_pipeline(StandardScaler(), SVC(kernel='rbf', gamma=gamma)).fit(Xmtr, ymtr)
    print(f'gamma={gamma}: train {mg.score(Xmtr, ymtr):.3f} / test {mg.score(Xmte, ymte):.3f}')

> **관찰:** 0.1 → 0.848/0.844(과소) · **1 → 0.957/0.956(적합)** · 100 → 0.990/0.944(과적합 시작) · **1000 → 1.000/0.500** — 점 하나하나를 섬처럼 감싼 경계가 시험에서 동전 수준으로 추락. **"train 만점은 경고"(M2a)의 가장 극적인 실물.** C·gamma는 교차검증으로(M2b).

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "A={1,3}, B={7,9}의 최적 경계·마진·SV를 손으로 구할 테니 채점해 줘."
- "비SV를 아무리 옮겨도 경계가 안 변하는 이유를 설명해 볼게 — 허점을 찔러 줘."
- "x² 접어 올리기가 커널의 정신인 이유를 설명해 볼게."
- "gamma=1000의 1.000/0.500을 M2a의 언어로 진단해 볼게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 1차원 세트피스로 **마진 최대화**(t=4=(2+6)/2)와 **SV의 지배력**(8→20 불변·6→5면 3.5)을 완주했다 — sklearn 일치
2. blobs에서 마진·SV(3개)와 **C의 효과**(0.1이면 SV 7개)를 봤다
3. **x² 접어 올리기**(0.571→1.0)로 커널의 정신을 재현하고, moons(rbf 0.956)·**gamma 극단**(1.000/0.500)을 확인했다

**스스로 점검**
- [ ] 후보 경계의 마진을 손으로 구할 수 있다
- [ ] SV의 정의와 지배력 실험을 설명할 수 있다
- [ ] C가 무엇의 벌점인지 안다
- [ ] x² 접어 올리기를 종이에 재현할 수 있다
- [ ] gamma=1000의 진단(극단 과적합)을 M2a로 말할 수 있다

**🔹심화 (선택)**
- Part D의 `noise`를 0.4로 키워 linear/rbf 격차가 어떻게 변하는지 보세요.
- `GridSearchCV`로 C·gamma를 함께 튜닝해 보세요(M2b의 자동화판).
- `decision_function` 값을 히스토그램으로 그려 "경계까지의 거리" 분포를 보세요.

**다음 시간(M8):** 정답 없는 세계 — K-means(묶기)와 PCA(줄이기).